# Zero Trust Certificate Lifecycle Lab Guide

This interactive notebook walks through the complete certificate lifecycle in a Zero Trust Architecture:

1. **PKI Hierarchy** — Explore the three independent PKI hierarchies (RSA, ECC, ML-DSA-87)
2. **Certificate Issuance** — Issue certificates via REST API, EST, and ACME
3. **Security Events** — Trigger compromise events via EDR/SIEM
4. **Automated Revocation** — Watch Event-Driven Ansible revoke certificates
5. **Verification** — Verify revocation via OCSP and CRL
6. **Monitoring** — Check Prometheus metrics and Grafana dashboards

## Prerequisites
Start the lab with `./start-lab.sh --all` before running this notebook.

In [ ]:
import httpx
import json
import time
from IPython.display import display, HTML, Markdown

# Lab configuration
EDR_URL = 'http://edr.cert-lab.local:8000'
SIEM_URL = 'http://siem.cert-lab.local:8000'
CT_LOG_URL = 'http://ct-log.cert-lab.local:8000'
CDP_URL = 'http://crl.cert-lab.local:8080'
POLICY_URL = 'http://policy.cert-lab.local:8000'
CHAIN_VIZ_URL = 'http://chain-viz.cert-lab.local:8000'

# Use localhost fallback for direct access
try:
    httpx.get(f'{EDR_URL}/health', timeout=3)
except Exception:
    EDR_URL = 'http://localhost:8082'
    SIEM_URL = 'http://localhost:8083'
    CT_LOG_URL = 'http://localhost:8086'
    CDP_URL = 'http://localhost:8088'
    POLICY_URL = 'http://localhost:8089'
    CHAIN_VIZ_URL = 'http://localhost:8090'

print('Lab Guide Ready')
print(f'EDR:  {EDR_URL}')
print(f'SIEM: {SIEM_URL}')

## Step 1: Check Lab Health

Verify all services are running before proceeding.

In [ ]:
services = {
    'Mock EDR': f'{EDR_URL}/health',
    'Mock SIEM': f'{SIEM_URL}/health',
    'CT Log': f'{CT_LOG_URL}/health',
    'CRL CDP': f'{CDP_URL}/health',
    'Policy Engine': f'{POLICY_URL}/health',
    'Chain Visualizer': f'{CHAIN_VIZ_URL}/health',
}

for name, url in services.items():
    try:
        r = httpx.get(url, timeout=5)
        status = 'UP' if r.status_code == 200 else f'HTTP {r.status_code}'
        icon = '\u2705' if r.status_code == 200 else '\u274c'
    except Exception as e:
        status = str(e)[:40]
        icon = '\u274c'
    print(f'{icon} {name}: {status}')

## Step 2: Explore PKI Hierarchies

The lab runs three independent PKI hierarchies with different cryptographic algorithms.
Each has: Root CA → Intermediate CA → IoT Sub-CA, plus EST RA, OCSP, and KRA.

In [ ]:
# Fetch trust chain data from the chain visualizer
try:
    r = httpx.get(f'{CHAIN_VIZ_URL}/api/chains', timeout=10)
    chains = r.json()
    
    for pki_type, info in chains.items():
        print(f'\n=== {info["label"]} ===')
        for node in info['nodes']:
            status = 'UP' if node.get('up') else 'DOWN'
            indent = '  ' if node.get('parent') else ''
            if node.get('parent') == 'intermediate':
                indent = '    '
            certs = ''
            if node.get('certs_valid'):
                certs = f' (valid: {node["certs_valid"]}, revoked: {node.get("certs_revoked", 0)})'
            print(f'{indent}{node["name"]}: {status}{certs}')
except Exception as e:
    print(f'Chain visualizer not available: {e}')
    print('Open http://localhost:8090 in your browser for the visual version')

## Step 3: Issue a Certificate

Issue a TLS certificate from the IoT Sub-CA using the Mock EDR's API.
This simulates an IoT device enrolling for its identity certificate.

In [ ]:
# Trigger certificate issuance via EDR
device_name = f'lab-guide-device-{int(time.time()) % 10000}'
device_fqdn = f'{device_name}.cert-lab.local'

print(f'Issuing certificate for: {device_fqdn}')
print(f'PKI Type: RSA-4096')
print(f'CA: IoT Sub-CA')

# Use the EDR to trigger a security event that will show in Kafka
event = {
    'event_type': 'malware_detection',
    'device_id': device_name,
    'device_fqdn': device_fqdn,
    'severity': 'high',
    'description': f'Lab guide demo - certificate lifecycle test for {device_fqdn}',
}

print(f'\nNote: To issue a certificate, run from the host:')
print(f'  lab issue --cn {device_fqdn} --pki-type rsa --ca iot')
print(f'\nOr use the pki-cli.py script:')
print(f'  ./scripts/pki-cli.py issue --ca iot --cn {device_fqdn}')

## Step 4: Validate Against Policy Engine

Before issuance, check if the certificate request complies with organizational policies.

In [ ]:
# Check policy compliance
request = {
    'common_name': device_fqdn,
    'organization': 'Cert-Lab',
    'country': 'US',
    'key_type': 'rsa',
    'key_size': 4096,
    'validity_days': 365,
    'cert_type': 'iot',
}

try:
    r = httpx.post(f'{POLICY_URL}/validate', json=request, timeout=10)
    result = r.json()
    
    if result['approved']:
        print(f'\u2705 Policy check APPROVED for {device_fqdn}')
    else:
        print(f'\u274c Policy check DENIED for {device_fqdn}')
    
    if result.get('violations'):
        print('\nViolations:')
        for v in result['violations']:
            print(f'  - [{v["rule"]}] {v["message"]}')
    
    if result.get('warnings'):
        print('\nWarnings:')
        for w in result['warnings']:
            print(f'  - [{w["rule"]}] {w["message"]}')
except Exception as e:
    print(f'Policy engine not available: {e}')

## Step 5: Trigger a Security Event

Simulate a key compromise detection. This sends an event to Kafka,
which EDA picks up and triggers automatic certificate revocation.

In [ ]:
# Send a security event via Mock EDR
# In the real flow: EDR -> Kafka -> EDA -> Ansible -> Dogtag revocation
event = {
    'event_type': 'key_compromise',
    'device_id': device_name,
    'severity': 'critical',
    'description': 'Private key material detected in code repository',
    'pki_type': 'rsa',
}

try:
    r = httpx.post(f'{EDR_URL}/events', json=event, timeout=10)
    if r.status_code == 200:
        data = r.json()
        print(f'\u2705 Security event published to Kafka')
        print(f'  Event ID:   {data.get("event_id", "N/A")}')
        print(f'  Type:       {event["event_type"]}')
        print(f'  Severity:   {event["severity"]}')
        print(f'  Device:     {device_name}')
        print(f'\nEDA will now process this event and trigger revocation...')
        print(f'Monitor with: podman logs -f eda-server')
    else:
        print(f'Error: {r.status_code} - {r.text}')
except Exception as e:
    print(f'EDR not available: {e}')
    print('Run the event manually: lab trigger --event-type key_compromise')

## Step 6: Check CRL Distribution Point

After revocation, the CRL is updated. The CDP server serves CRLs over HTTP
per RFC 5280 Section 4.2.1.13.

In [ ]:
# Check CDP server status
try:
    r = httpx.get(f'{CDP_URL}/crl/status.json', timeout=10)
    if r.status_code == 200:
        data = r.json()
        print(f'CRL Distribution Point Server')
        print(f'  Last refresh:  {data.get("last_refresh", "N/A")}')
        print(f'  CAs reached:   {data.get("cas_success", 0)}/{data.get("cas_total", 0)}')
        print(f'  Refresh rate:  every {data.get("refresh_interval", 300)}s')
        
        # List available CRLs
        crl_r = httpx.get(f'{CDP_URL}/', timeout=10)
        if crl_r.status_code == 200:
            files = crl_r.json()
            crl_files = [f['name'] for f in files if f.get('name', '').endswith('.crl')]
            print(f'\nAvailable CRLs ({len(crl_files)}):')
            for f in crl_files:
                print(f'  {CDP_URL}/crl/{f}')
except Exception as e:
    print(f'CDP server not available: {e}')

## Step 7: CT Log Verification

Check the Certificate Transparency log for all issued certificates.
The CT log maintains a Merkle tree of all certificates (RFC 6962).

In [ ]:
try:
    r = httpx.get(f'{CT_LOG_URL}/stats', timeout=10)
    if r.status_code == 200:
        data = r.json()
        print(f'Certificate Transparency Log')
        print(f'  Log ID:     {data.get("log_id", "N/A")}')
        print(f'  Tree size:  {data.get("tree_size", 0)} entries')
        print(f'  Root hash:  {data.get("root_hash", "N/A")[:32]}...')
        
        if data.get('entries_by_pki'):
            print(f'\n  Entries by PKI type:')
            for pki, count in data['entries_by_pki'].items():
                print(f'    {pki}: {count}')
except Exception as e:
    print(f'CT log not available: {e}')

## Step 8: Monitoring

Check PKI metrics from Prometheus. The monitoring stack tracks:
- CA health and certificate inventory
- OCSP response times
- CRL update status
- Revocation rates

**Grafana Dashboard:** http://localhost:3000 (admin / see .env)

In [ ]:
# Query Prometheus for PKI metrics
PROM_URL = 'http://prometheus.cert-lab.local:9090'
try:
    httpx.get(f'{PROM_URL}/-/healthy', timeout=3)
except Exception:
    PROM_URL = 'http://localhost:9090'

queries = {
    'CAs Up': 'sum(pki_ca_up)',
    'Valid Certs': 'sum(pki_certificates_total{status="VALID"})',
    'Revoked Certs': 'sum(pki_certificates_total{status="REVOKED"})',
    'CDP Status': 'pki_cdp_server_up',
    'CT Log Size': 'pki_ct_log_tree_size',
}

try:
    for label, query in queries.items():
        r = httpx.get(f'{PROM_URL}/api/v1/query', params={'query': query}, timeout=10)
        if r.status_code == 200:
            data = r.json()
            results = data.get('data', {}).get('result', [])
            if results:
                value = results[0].get('value', [None, 'N/A'])[1]
                print(f'  {label}: {value}')
            else:
                print(f'  {label}: no data')
except Exception as e:
    print(f'Prometheus not available: {e}')
    print('Open http://localhost:9090 for direct access')

## Summary

This lab demonstrates the complete Zero Trust certificate lifecycle:

| Step | Component | Protocol/Standard |
|------|-----------|------------------|
| Policy Check | Policy Engine | CA/B Forum BR, RFC 5280 |
| Issuance | Dogtag PKI | CMC (RFC 5272), EST (RFC 7030), ACME (RFC 8555) |
| Transparency | CT Log | RFC 6962 |
| Detection | EDR/SIEM → Kafka | Event-driven security |
| Revocation | EDA → Ansible → Dogtag | CRL (RFC 5280), OCSP (RFC 6960) |
| Distribution | CDP Server | RFC 5280 Section 4.2.1.13 |
| Monitoring | Prometheus + Grafana | Metrics-based observability |

### Key URLs
- **Grafana:** http://localhost:3000
- **Prometheus:** http://localhost:9090
- **Chain Visualizer:** http://localhost:8090
- **CDP Server:** http://localhost:8088
- **Jupyter:** http://localhost:8888

### Next Steps
- Try `lab test --all` to run all 26 security scenarios
- Run `lab est-enroll` for EST protocol enrollment
- Run `lab acme-issue` for ACME protocol enrollment
- Explore cross-certification: `sudo bash scripts/pki/cross-certify.sh rsa-ecc`
- Run compliance scan: `python scripts/compliance-scan.py --all`